# Results for model: institute-of-science-tokyo/llama-3.1-swallow-8b-instruct-v0.1

In [ ]:
import polars as pl
import xgboost as xgb
import numpy as np

# Load data
df = pl.read_parquet('jane_street_data/train.parquet')

# Feature Engineering
df = df.with_column(
    pl.when(df['responder_6'] > df['feature_00']).then(df['feature_00']).otherwise(pl.lit(0)).alias('feature_00')
)

df = df.with_column(
    pl.rolling(df, 'date_id', 14).apply(lambda x: x.quantile(0.85)).alias('top_quantile')
)

df = df.with_column(
    pl.when(df['feature_00'] > df['top_quantile']).then(1).otherwise(0).alias('feature_01')
)

# Train XGBoost Regressor
X = df['feature_01'].to_numpy()
y = df['responder_6'].to_numpy()

xgb_model = xgb.XGBRegressor(objective='reg:squarederror', max_depth=5, learning_rate=0.1, n_estimators=100)
xgb_model.fit(X, y)